# 🔍 Making a Prediction — Step by Step

This notebook walks through **one full prediction** so you can see exactly what the system does.

No GPU required — we use BM25, a keyword-based search algorithm.

## The pipeline

```
STEP 1  Load a real conversation from the dev set
           │
STEP 2  Build a query string from the conversation history
           │
STEP 3  BM25 searches 47,071 tracks for the best keyword matches
           │  returns top-20 track IDs
STEP 4  Compare our top-20 to the ground truth
           │
STEP 5  (Optional) LLM writes a response — requires GPU / Llama access
```

### What is BM25?

BM25 is like a very fast library index:
1. Pre-indexes every track by its name, artist, album words
2. Takes your query (the conversation text)
3. Returns tracks whose words overlap most with the query

No neural network, no GPU. Despite being simple, it outperforms BERT on this dataset!

In [1]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import math
import os
import pandas as pd
from datasets import load_dataset
from IPython.display import display, HTML

pd.set_option("display.max_colwidth", 100)
print("Ready!")

Ready!


---
## Step 1 — Load a real conversation from the dev set

In [2]:
# Change these to explore different sessions / turns
TARGET_TURN = 3
SESSION_IDX = 0

print("Loading dev set...")
conv_test = load_dataset("talkpl-ai/TalkPlayData-Challenge-Dataset", split="test")
print(f"Dev set: {len(conv_test):,} sessions")

session = conv_test[SESSION_IDX]
profile = session.get("user_profile", {})
goal = session.get("conversation_goal", {})

print(f"\n─── Session {SESSION_IDX} ────────────────────────")
print(f"User:    {profile.get('age_group','?')} {profile.get('gender','?')} from {profile.get('country_name','?')}")
print(f"Culture: {profile.get('preferred_musical_culture','?')}")
print(f"Goal:    {goal.get('listener_goal','?')[:160]}")
print(f"\nPredicting the song for turn {TARGET_TURN}.")

Loading dev set...


Dev set: 1,000 sessions

─── Session 0 ────────────────────────
User:    30s male from Mexico
Culture: Anglo-American Rock
Goal:    play one specific song that is known for its high popularity within its genre or era.

Predicting the song for turn 3.


In [3]:
def display_conversation(session, target_turn=None, track_meta=None):
    """Dark-mode-friendly conversation renderer."""
    rows = []
    for turn in session["conversations"]:
        role = turn["role"]
        content = turn["content"]
        tnum = turn["turn_number"]
        is_target = tnum == target_turn

        if role == "user":
            opacity = "0.22" if is_target else "0.12"
            border_w = "3px" if is_target else "2px"
            label = f"Turn {tnum} · User" + (" ← selected" if is_target else "")
            rows.append(
                f'<tr><td style="background:rgba(59,130,246,{opacity});border-left:{border_w} solid #3b82f6;'
                f'text-align:right;padding:8px 12px;border-radius:0 8px 8px 0">'
                f'<small><b>{label}</b></small><br>{content}</td></tr>'
            )
        elif role == "assistant":
            rows.append(
                f'<tr><td style="background:rgba(34,197,94,0.12);border-left:2px solid #22c55e;'
                f'text-align:left;padding:8px 12px;border-radius:0 8px 8px 0">'
                f'<small><b>Turn {tnum} · Assistant</b></small><br>{content}</td></tr>'
            )
        elif role == "music":
            if track_meta and content in track_meta:
                meta = track_meta[content]
                name = (meta.get("track_name") or ["?"])[0]
                artist = (meta.get("artist_name") or ["?"])[0]
                tags = ", ".join((meta.get("tag_list") or [])[:4])
                content_html = f"<b>{name}</b> — {artist}<br><small style='opacity:0.7'>{tags}</small>"
            else:
                content_html = f"<small style='opacity:0.6'>track_id: {content}</small>"
            opacity = "0.22" if is_target else "0.12"
            note = " ← PREDICT THIS" if is_target else ""
            rows.append(
                f'<tr><td style="background:rgba(202,138,4,{opacity});border:1px dashed rgba(202,138,4,0.7);'
                f'text-align:center;padding:8px 12px;border-radius:8px">'
                f'🎵 <small><b>Ground Truth · Turn {tnum}{note}</b></small><br>{content_html}</td></tr>'
            )

    table = "<table style='width:100%;border-collapse:separate;border-spacing:0 4px'>" + "".join(rows) + "</table>"
    display(HTML(table))

display_conversation(session, target_turn=TARGET_TURN)

"Turn 1 · UserPlay the highly popular grunge track from the 90s, 'Heart-Shaped Box' by Nirvana."
🎵 Ground Truth · Turn 1track_id: 2445ed62-2e19-4222-8d01-3a57f685755d
"Turn 1 · AssistantAbsolutely! Pulling up ""Heart-Shaped Box"" by Nirvana for you right now. Great choice from the 90s!"
"Turn 2 · UserPerfect! That was the popular song I was looking for. Now, what are some other highly popular alternative rock tracks you have?"
🎵 Ground Truth · Turn 2track_id: 9fe2712d-337b-4b9a-9a75-285b89e1543b
"Turn 2 · AssistantGlad you liked that one! If you're looking for more popular alternative rock, how about ""Fluorescent Adolescent"" by Arctic Monkeys? It's another highly-rated track with a great alt-rock vibe."
"Turn 3 · User ← selectedYes, that's a great one! Arctic Monkeys always deliver. What other highly popular alternative rock tracks, maybe something with a bit more of a post-punk revival feel, do you have?"
🎵 Ground Truth · Turn 3 ← PREDICT THIStrack_id: 3d02bc2d-7c9a-4ea2-9361-b05b2fe654c1
"Turn 3 · AssistantYou got it! Since you're enjoying Arctic Monkeys and looking for that post-punk revival vibe, ""D Is For Dangerous"" is another excellent choice from them. It definitely delivers on that energetic, alternative rock sound."
"Turn 4 · UserAnother solid track from Arctic Monkeys. I appreciate the focus on their sound. Can you recommend another highly popular alternative rock song, perhaps by a different band that also has that energetic, modern rock feel?"
🎵 Ground Truth · Turn 4track_id: dfcc86b1-610b-4776-8084-dc2b3726e980


---
## Step 2 — Build the query string

BM25 takes plain text. We concatenate all turns *before* the target turn into one string.

In [4]:
def build_query(conversations, target_turn_number):
    """Concatenate user + assistant turns before the target turn, plus the target user message."""
    parts = []
    for turn in conversations:
        if turn["turn_number"] < target_turn_number and turn["role"] in ("user", "assistant"):
            parts.append(f"{turn['role']}: {turn['content']}")
    for turn in conversations:
        if turn["turn_number"] == target_turn_number and turn["role"] == "user":
            parts.append(f"user: {turn['content']}")
            break
    return "\n".join(parts)

query = build_query(session["conversations"], TARGET_TURN)

print("══ Query passed to BM25 ══")
print(query)
print(f"\n({len(query.split())} words)")

══ Query passed to BM25 ══
user: Play the highly popular grunge track from the 90s, 'Heart-Shaped Box' by Nirvana.
assistant: Absolutely! Pulling up "Heart-Shaped Box" by Nirvana for you right now. Great choice from the 90s!
user: Perfect! That was the popular song I was looking for. Now, what are some other highly popular alternative rock tracks you have?
assistant: Glad you liked that one! If you're looking for more popular alternative rock, how about "Fluorescent Adolescent" by Arctic Monkeys? It's another highly-rated track with a great alt-rock vibe.
user: Yes, that's a great one! Arctic Monkeys always deliver. What other highly popular alternative rock tracks, maybe something with a bit more of a post-punk revival feel, do you have?

(115 words)


---
## Step 3 — BM25 Retrieval

> **First run:** Building the index takes ~1–2 minutes. It's automatically saved to `../cache/` — future runs are instant.

In [5]:
from mcrs.retrieval_modules.bm25 import BM25_MODEL

print("Loading BM25 index (first run builds and caches it — ~1-2 min)...")
bm25 = BM25_MODEL(
    dataset_name="talkpl-ai/TalkPlayData-Challenge-Track-Metadata",
    split_types=["all_tracks"],
    corpus_types=["track_name", "artist_name", "album_name"],
    cache_dir="../cache"
)
print(f"BM25 ready — {len(bm25.track_ids):,} tracks indexed.")

Loading BM25 index (first run builds and caches it — ~1-2 min)...


BM25 ready — 47,071 tracks indexed.


In [6]:
top20_ids = bm25.text_to_item_retrieval(query, topk=20)
print(f"Retrieved {len(top20_ids)} tracks.\n")

rows = []
for rank, tid in enumerate(top20_ids, start=1):
    meta = bm25.metadata_dict.get(tid, {})
    rows.append({
        "Rank":       rank,
        "Track Name": (meta.get("track_name")  or ["?"])[0],
        "Artist":     (meta.get("artist_name") or ["?"])[0],
        "Album":      (meta.get("album_name")  or ["?"])[0],
        "Tags":       ", ".join((meta.get("tag_list") or [])[:4]),
        "Popularity": meta.get("popularity", "?"),
    })

display(pd.DataFrame(rows))

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved 20 tracks.



,Rank,Track Name,Artist,Album,Tags,Popularity
0,1,Heart-Shaped Box,Dead Sara,Heart-Shaped Box,"hard rock, cover, cover songs, covers",51.0
1,2,What If You Were Right The First Time?,Arctic Monkeys,Brianstorm,"punk, rock, Indie Rock, Rock",32.0
2,3,So Into You,Atlanta Rhythm Section,A Rock And Roll Alternative,"polski hip hop, hifi banda, hades rakraczej, hip hop",64.0
3,4,You Get What You Give,New Radicals,Maybe You've Been Brainwashed Too,"a dynamic male vocalist, energetic, loved, 1998",80.0
4,5,Fluorescent Adolescent,Arctic Monkeys,Favourite Worst Nightmare,"punk, beautiful, fucking fuck, great lyrics",74.0
5,6,You Got Me Singing,Leonard Cohen,Popular Problems,"Pop, fip, singer-songwriter",47.0
6,7,Did I Ever Love You,Leonard Cohen,Popular Problems,"2014 single, canadian, 2014, country",43.0
7,8,Find What You're Looking For,Olivia O'Brien,Find What You're Looking For,"pop, Pop, 2016 single",48.0
8,9,Popular Song,MIKA,Yours Truly,"hard rock, top100:2013, makes me laugh, pop-rap",58.0
9,10,Heart-Shaped Box,Nirvana,In Utero,"Indie Rock, Rock, Alternative, grunge",84.0


---
## Step 4 — Compare with the ground truth

In [7]:
ground_truth_id = None
for turn in session["conversations"]:
    if turn["turn_number"] == TARGET_TURN and turn["role"] == "music":
        ground_truth_id = turn["content"]
        break

if ground_truth_id:
    gt = bm25.metadata_dict.get(ground_truth_id, {})
    print("══ Ground Truth Track ══")
    print(f"  Track:  {(gt.get('track_name')  or ['?'])[0]}")
    print(f"  Artist: {(gt.get('artist_name') or ['?'])[0]}")
    print(f"  Album:  {(gt.get('album_name')  or ['?'])[0]}")
    print(f"  Tags:   {', '.join((gt.get('tag_list') or [])[:5])}")
    print()
    if ground_truth_id in top20_ids:
        rank = top20_ids.index(ground_truth_id) + 1
        ndcg = 1.0 / math.log2(rank + 1)
        print(f"  ✅ FOUND at rank {rank}  (NDCG contribution: {ndcg:.4f})")
    else:
        print("  ❌ NOT in our top-20.")
        print("     (NDCG@10 for BM25 is only 0.0627 overall — misses are common)")

══ Ground Truth Track ══
  Track:  D Is For Dangerous
  Artist: Arctic Monkeys
  Album:  Favourite Worst Nightmare
  Tags:   punk, i could dance to this one, favorite, post-punk revival, great songs

  ❌ NOT in our top-20.
     (NDCG@10 for BM25 is only 0.0627 overall — misses are common)


In [8]:
# Highlight the ground truth row in the results table
if ground_truth_id and ground_truth_id in top20_ids:
    gt_rank = top20_ids.index(ground_truth_id) + 1

    def highlight_gt(row):
        if row["Rank"] == gt_rank:
            return ["background-color: rgba(202,138,4,0.25)"] * len(row)
        return [""] * len(row)

    display(pd.DataFrame(rows).style.apply(highlight_gt, axis=1))
    print("(Highlighted row = ground truth track)")
else:
    print("Ground truth not in top-20 — no row to highlight.")

Ground truth not in top-20 — no row to highlight.


---
## Step 5 — LLM Response Generation (optional)

In the full system, the top-1 retrieved track is passed to **Llama-3.2-1B-Instruct** which writes a friendly explanation. Requires GPU + HuggingFace auth.

Below we show what the inputs look like, and you can set `RUN_LLM = True` to actually run it.

In [9]:
# Show the system prompt components
prompt_dir = "../mcrs/system_prompts"
for fname in ["roleplay.txt", "response_generation.txt", "personalization.txt"]:
    path = os.path.join(prompt_dir, fname)
    if os.path.exists(path):
        print(f"── {fname} ──")
        print(open(path).read())
        print()

── roleplay.txt ──
You are an expert music recommendation assistant. Your task is to understand user preferences and provide personalized music recommendations.


── response_generation.txt ──
Based on the user query and the recommended track from tool calling results, provide a brief response that:
1. MUST base your response on the previously recommended track from the tool calling results
2. If the recommended track doesn't match the user's query, apologize and acknowledge the mismatch. It is problem of recommendation system.
3. If it's a good match, acknowledge that you've found music that matches their request with enthusiasm and confidence
4. Share key details including title, artist, and relevant musical information (genre, mood, style, or notable characteristics)
5. Briefly explain why this track is a good match for their specific request or preferences (or apologize if it's not a good match)
6. Invite further interaction by asking if they'd like to explore similar tracks, need 

In [10]:
top1_id = top20_ids[0]
top1 = bm25.metadata_dict.get(top1_id, {})
recommend_str = (
    f"track_id: {top1_id}, "
    f"track_name: {(top1.get('track_name') or ['?'])[0].lower()}, "
    f"artist_name: {(top1.get('artist_name') or ['?'])[0].lower()}, "
    f"album_name: {(top1.get('album_name') or ['?'])[0].lower()}, "
    f"release_date: {top1.get('release_date','?')}"
)
print("── Item string fed to LLM ──")
print(recommend_str)

── Item string fed to LLM ──
track_id: a8ff5da0-a9ce-4e97-8ae4-12e7acb5e463, track_name: heart-shaped box, artist_name: dead sara, album_name: heart-shaped box, release_date: 2013-11-12


In [11]:
RUN_LLM = False  # Set True if you have GPU + HuggingFace access

if RUN_LLM:
    try:
        import torch
        from mcrs import load_crs_baseline
        crs = load_crs_baseline(
            lm_type="meta-llama/Llama-3.2-1B-Instruct",
            retrieval_type="bm25",
            cache_dir="../cache",
            device="mps",   # "cuda" on Linux, "cpu" if no GPU
            dtype=torch.float32,
            attn_implementation="eager"
        )
        result = crs.chat(query, user_id=session["user_id"])
        print(result["response"])
    except Exception as e:
        print(f"LLM unavailable: {e}")
else:
    name = (top1.get('track_name') or ['?'])[0]
    artist = (top1.get('artist_name') or ['?'])[0]
    print("RUN_LLM = False — showing a sample response:\n")
    print(f"Based on our conversation, I'd recommend '{name}' by {artist}. "
          f"This track fits the mood perfectly — would you like something similar, "
          f"or shall we explore a different direction?")

RUN_LLM = False — showing a sample response:

Based on our conversation, I'd recommend 'Heart-Shaped Box' by Dead Sara. This track fits the mood perfectly — would you like something similar, or shall we explore a different direction?


---
## Experiment: Try your own query

In [12]:
custom_query = "upbeat electronic dance music with a lot of energy"

custom_ids = bm25.text_to_item_retrieval(custom_query, topk=10)
print(f"Query: '{custom_query}'\n")
rows = []
for i, tid in enumerate(custom_ids, 1):
    meta = bm25.metadata_dict.get(tid, {})
    rows.append({
        "Rank":   i,
        "Track":  (meta.get("track_name")  or ["?"])[0],
        "Artist": (meta.get("artist_name") or ["?"])[0],
        "Tags":   ", ".join((meta.get("tag_list") or [])[:5]),
        "Pop":    meta.get("popularity", "?"),
    })
display(pd.DataFrame(rows))

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query: 'upbeat electronic dance music with a lot of energy'



,Rank,Track,Artist,Tags,Pop
0,1,Shake It,Elephant Man,"lady saw, Dance, mr vegas, popcaan, sick head",10.0
1,2,Party Up In Here,Elephant Man,"Dancehall, Dance, Reggae, exercise video",13.0
2,3,Evum,Plej,"lounge at home two, chill chill, lounge electronic, minimal, acid lounge",44.0
3,4,Dance to the Music,Sly & The Family Stone,"rollingstone 500 greatest songs of all time, fantastic opener, rhythm and blues, funk tag, freedom",59.0
4,5,Dance and Sweep,Elephant Man,"good dancehall, Dance, Dancehall, dancehall, shake your hips",25.0
5,6,Getting Away with It - 2013 Remaster,Electronic,"supergroup, johnny marr, loved, a vocal-centric aesthetic, pretty",49.0
6,7,Energy,Elektronimia,"Dance, Electronic",1.0
7,8,Express Yourself,Labrinth,"loved, makes me smile, simple, 2012 singles, check",42.0
8,9,Music That You Can Dance To,Sparks,"Dance, liked it, can, incessant, trance",32.0
9,10,Step Ova,Elephant Man,"Dancehall, Dance, Reggae",14.0


In [13]:
example_queries = [
    "sad acoustic guitar breakup song",
    "80s synthwave retro neon",
    "hip hop workout motivation",
    "classical piano relaxing study",
]
for q in example_queries:
    ids = bm25.text_to_item_retrieval(q, topk=3)
    print(f"\n'{q}'")
    for i, tid in enumerate(ids, 1):
        meta = bm25.metadata_dict.get(tid, {})
        name   = (meta.get("track_name")  or ["?"])[0]
        artist = (meta.get("artist_name") or ["?"])[0]
        tags   = ", ".join((meta.get("tag_list") or [])[:3])
        print(f"  {i}. {name} — {artist}  [{tags}]")

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


'sad acoustic guitar breakup song'
  1. Potential Breakup Song — Aly & AJ  [energetic, alternative rock, it is about time this song was written and recorded]
  2. Sad Song (feat. Elena Coates) — Elena Coats  [linedance 2016, Alternative, Pop]
  3. Macon — Jamey Johnson  [lol  whooo, grammy, country]


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


'hip hop workout motivation'
  1. Hip Hop — Dead Prez  [socially conscious, east coast rap, energetic]
  2. 808 — Blaque  [billboard top 20, forever stuck on repeat, 90s]
  3. Hip Hop Sinister — Hopsin  [Hip-Hop/Rap, west coast rap, underground hip hop]


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


'classical piano relaxing study'
  1. Gymnopedie — Richard Mollenbeck  [New Age, Christian]
  2. Adagio Albinoni — Richard Mollenbeck  [New Age, Christian]
  3. Pilates Music 4 — Meditation Spa  [newage, New Age]


---
## What we just did

1. Loaded a real conversation and built a query string
2. BM25 searched 47,071 tracks and returned the top-20
3. Compared our prediction to the ground truth

The system scores NDCG@10 = 0.0627 overall — there's a lot of room for improvement!

**Next:** Open `03_evaluate_predictions.ipynb` to browse results interactively across the whole dev set.